# 🏥 Medical Insurance Charges — Linear Regression Analysis

> **Goal:** Predict individual medical insurance charges using demographic and lifestyle features.  
> **Dataset:** 1,338 records · 6 features · sourced from [Kaggle – Medical Cost Personal Dataset](https://www.kaggle.com/datasets/mirichoi0218/insurance)  
> **Author:** *(your name)*  

---

## Table of Contents
1. [Setup & Imports](#1-setup--imports)
2. [Data Loading & Overview](#2-data-loading--overview)
3. [Exploratory Data Analysis (EDA)](#3-exploratory-data-analysis-eda)
4. [Feature Engineering & Preprocessing](#4-feature-engineering--preprocessing)
5. [Model Training](#5-model-training)
6. [Model Evaluation](#6-model-evaluation)
7. [Residual Analysis](#7-residual-analysis)
8. [Feature Importance (Coefficients)](#8-feature-importance-coefficients)
9. [Key Takeaways](#9-key-takeaways)

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

# Consistent aesthetics
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

RANDOM_STATE = 42
print('Libraries loaded ✅')

## 2. Data Loading & Overview

In [ ]:
df = pd.read_csv('insurance.csv')

print(f'Shape: {df.shape}')
print(f'Duplicates: {df.duplicated().sum()}')
print(f'Missing values:\n{df.isnull().sum()}')
df.head()

In [ ]:
# Drop duplicate rows if any
df = df.drop_duplicates()
print(f'Shape after deduplication: {df.shape}')
df.describe().round(2)

## 3. Exploratory Data Analysis (EDA)

We examine distributions, correlations, and how each feature relates to insurance charges.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Feature Distributions', fontsize=16, fontweight='bold')

num_cols = ['age', 'bmi', 'children', 'charges']
cat_cols = ['sex', 'smoker', 'region']

for i, col in enumerate(['age', 'bmi', 'charges']):
    ax = axes[0, i]
    df[col].hist(ax=ax, bins=30, color='steelblue', edgecolor='white')
    ax.set_title(col.capitalize())
    ax.set_xlabel(col)

for i, col in enumerate(cat_cols):
    ax = axes[1, i]
    df[col].value_counts().plot(kind='bar', ax=ax, color='coral', edgecolor='white', rot=0)
    ax.set_title(col.capitalize())

plt.tight_layout()
plt.show()

In [ ]:
# Charges vs BMI — colored by smoker status
plt.figure(figsize=(9, 5))
sns.scatterplot(data=df, x='bmi', y='charges', hue='smoker',
                palette={'yes': '#e74c3c', 'no': '#3498db'}, alpha=0.7)
plt.title('BMI vs Insurance Charges (by Smoking Status)', fontsize=13)
plt.xlabel('BMI')
plt.ylabel('Charges (USD)')
plt.legend(title='Smoker')
plt.tight_layout()
plt.show()
print('⚠️  Smokers clearly form a separate, higher-charge cluster — an important signal for the model.')

In [ ]:
# Charges vs Age — colored by smoker
plt.figure(figsize=(9, 5))
sns.scatterplot(data=df, x='age', y='charges', hue='smoker',
                palette={'yes': '#e74c3c', 'no': '#3498db'}, alpha=0.7)
plt.title('Age vs Insurance Charges (by Smoking Status)', fontsize=13)
plt.xlabel('Age')
plt.ylabel('Charges (USD)')
plt.legend(title='Smoker')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
df_encoded = df.copy()
df_encoded['sex']    = df_encoded['sex'].map({'female': 1, 'male': 0})
df_encoded['smoker'] = df_encoded['smoker'].map({'yes': 1, 'no': 0})
df_encoded['region'] = df_encoded['region'].astype('category').cat.codes

plt.figure(figsize=(8, 6))
sns.heatmap(df_encoded.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()
print('smoker has the highest correlation with charges (≈0.79)')

## 4. Feature Engineering & Preprocessing

- Binary-encode `sex` and `smoker`.
- Drop `region` (low correlation, would need one-hot encoding for multi-class).
- Add an interaction term `smoker × bmi` — smokers with high BMI face the highest charges.

In [ ]:
X = df.drop(columns=['charges', 'region']).copy()
y = df['charges']

X['sex']    = X['sex'].map({'female': 1, 'male': 0})
X['smoker'] = X['smoker'].map({'yes': 1, 'no': 0})

# Interaction feature: smoker × bmi
X['smoker_bmi'] = X['smoker'] * X['bmi']

print('Features used:', list(X.columns))
X.head()

## 5. Model Training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)

print(f'Train size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}')

model = LinearRegression()
model.fit(X_train, y_train)
print('Model trained ✅')

## 6. Model Evaluation

In [ ]:
y_pred = model.predict(X_test)

r2   = r2_score(y_test, y_pred)
n, p = X_test.shape
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# 5-fold cross-validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')

print('=' * 40)
print(f'  R²            : {r2:.4f}')
print(f'  Adjusted R²   : {adj_r2:.4f}')
print(f'  MAE           : ${mae:,.2f}')
print(f'  RMSE          : ${rmse:,.2f}')
print(f'  CV R² (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print('=' * 40)

## 7. Residual Analysis

A good linear model should have residuals randomly scattered around zero with no pattern.

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred, alpha=0.5, color='steelblue', edgecolors='white', linewidth=0.3)
lim = max(y_test.max(), y_pred.max())
axes[0].plot([0, lim], [0, lim], 'r--', linewidth=1.5, label='Perfect fit')
axes[0].set_xlabel('Actual Charges')
axes[0].set_ylabel('Predicted Charges')
axes[0].set_title('Actual vs Predicted')
axes[0].legend()

# Residuals vs Predicted
axes[1].scatter(y_pred, residuals, alpha=0.5, color='coral', edgecolors='white', linewidth=0.3)
axes[1].axhline(0, color='black', linewidth=1.2, linestyle='--')
axes[1].set_xlabel('Predicted Charges')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residuals vs Predicted')

plt.suptitle('Model Diagnostics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('Note: Heteroscedasticity visible at higher predictions — expected for a linear model on this dataset.')

## 8. Feature Importance (Model Coefficients)

In [ ]:
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=True)

plt.figure(figsize=(8, 5))
colors = ['#e74c3c' if c > 0 else '#3498db' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Linear Regression Coefficients\n(magnitude = importance)', fontsize=13)
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.show()

print(coef_df.to_string(index=False))

## 9. Key Takeaways

| Insight | Detail |
|---|---|
| **Smoking is the #1 driver** | The `smoker` and `smoker_bmi` interaction dominate the coefficients |
| **Model R² ≈ 0.86** | ~86 % of variance in charges explained — solid for a linear baseline |
| **Heteroscedasticity** | Residuals fan out at higher predictions; a log-transform on `charges` or a tree-based model could improve this |
| **Next steps** | Try Ridge/Lasso regularisation, log-transform the target, or gradient boosting (XGBoost) for non-linear interactions |

---
*Notebook built for portfolio demonstration · feel free to fork and extend!*